# 02 — Exploratory Data Analysis

**Purpose**: Answer the 4 required EDA questions with visualizations.

**Questions**:
1. What is the highest rated movie in 2023? Balance ratings vs votes.
2. Who is the most popular actor in 2023?
3. What are the user-movie preference trends over the years?
4. Define a hit movie metric (business justification).

**Prerequisites**: Run `01_data_loading.ipynb` first to generate `data/movies_clean.parquet`.

**Key decisions**:
- D008/D009: Actor popularity weighted by billing order, numVotes, rating
- D010: Min vote threshold configurable (default 500)
- D011: Hit metric two-mode (Full with ROI, Partial without)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Plotting config
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Paths
DATA_DIR = os.path.join('..', 'data')
FIGURES_DIR = os.path.join('..', 'reports', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Config (D010)
MIN_VOTE_THRESHOLD = 500

print('Loading data...')
movies = pd.read_parquet(os.path.join(DATA_DIR, 'movies_clean.parquet'))
actor_popularity = pd.read_parquet(os.path.join(DATA_DIR, 'actor_popularity.parquet'))
director_stats = pd.read_parquet(os.path.join(DATA_DIR, 'director_stats.parquet'))
print(f'  Movies: {len(movies):,} rows')
print(f'  Actors: {len(actor_popularity):,} rows')
print(f'  Directors: {len(director_stats):,} rows')

## Q1: Highest Rated Movie in 2023 + Ratings vs Votes Balance

**Approach**:
- Filter 2023 movies with >= MIN_VOTE_THRESHOLD votes
- Compute IMDB weighted rating (Bayesian average)
- Scatter plot: rating vs numVotes (log scale)
- Table: Top 10 by weighted rating

**Weighted Rating Formula** (IMDB Bayesian):
$$WR = \frac{v}{v+m} \cdot R + \frac{m}{v+m} \cdot C$$

Where:
- $R$ = movie's average rating
- $v$ = number of votes for the movie
- $m$ = minimum votes required (25th percentile)
- $C$ = mean rating across all movies

In [ ]:
# Filter 2023 movies
movies_2023 = movies[movies['startYear'] == 2023].copy()
print(f'2023 movies with >= {MIN_VOTE_THRESHOLD} votes: {len(movies_2023):,}')

# Compute weighted rating parameters
C = movies_2023['averageRating'].mean()  # Mean rating across all 2023 movies
m = movies_2023['numVotes'].quantile(0.25)  # 25th percentile of votes

print(f'\nWeighted Rating Parameters:')
print(f'  C (mean rating) = {C:.2f}')
print(f'  m (min votes, 25th pctl) = {m:.0f}')

# Compute weighted rating
v = movies_2023['numVotes']
R = movies_2023['averageRating']
movies_2023['weighted_rating'] = (v / (v + m)) * R + (m / (v + m)) * C

print(f'\nWeighted rating range: {movies_2023["weighted_rating"].min():.2f} - {movies_2023["weighted_rating"].max():.2f}')

In [ ]:
# Scatter plot: Rating vs NumVotes
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Raw rating vs votes
ax1 = axes[0]
scatter = ax1.scatter(
    movies_2023['numVotes'], 
    movies_2023['averageRating'],
    c=movies_2023['weighted_rating'],
    cmap='YlOrRd', alpha=0.6, edgecolors='gray', linewidth=0.5
)
ax1.set_xscale('log')
ax1.set_xlabel('Number of Votes (log scale)')
ax1.set_ylabel('Average Rating')
ax1.set_title('Q1: 2023 Movies — Rating vs Votes\n(color = weighted rating)')
plt.colorbar(scatter, ax=ax1, label='Weighted Rating')

# Annotate top 5 by weighted rating
top5 = movies_2023.nlargest(5, 'weighted_rating')
for _, row in top5.iterrows():
    ax1.annotate(
        row['primaryTitle'][:20], 
        (row['numVotes'], row['averageRating']),
        fontsize=8, ha='left', va='bottom'
    )

# Right: Distribution of weighted ratings
ax2 = axes[1]
ax2.hist(movies_2023['weighted_rating'], bins=30, color='steelblue', edgecolor='white')
ax2.axvline(movies_2023['weighted_rating'].mean(), color='red', linestyle='--', label=f'Mean: {movies_2023["weighted_rating"].mean():.2f}')
ax2.set_xlabel('Weighted Rating')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Weighted Ratings (2023)')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'q1_rating_vs_votes_2023.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q1_rating_vs_votes_2023.png')

In [ ]:
# Top 10 by weighted rating
print('\nTop 10 Highest Weighted Rating Movies (2023):')
print('='*80)
top10_weighted = movies_2023.nlargest(10, 'weighted_rating')[
    ['primaryTitle', 'averageRating', 'numVotes', 'weighted_rating', 'director_name', 'genres']
]
display(top10_weighted)

# Top 10 by raw rating (for comparison)
print('\nTop 10 Highest Raw Rating Movies (2023) — for comparison:')
print('='*80)
top10_raw = movies_2023.nlargest(10, 'averageRating')[
    ['primaryTitle', 'averageRating', 'numVotes', 'weighted_rating', 'director_name']
]
display(top10_raw)

In [ ]:
# Insight: High rating but low votes (hidden gems vs blockbusters)
print('\nInsight: Movies with High Rating but Low Votes (potential hidden gems):')
high_rating_low_votes = movies_2023[
    (movies_2023['averageRating'] >= 7.5) & 
    (movies_2023['numVotes'] < movies_2023['numVotes'].median())
].nlargest(5, 'averageRating')
display(high_rating_low_votes[['primaryTitle', 'averageRating', 'numVotes', 'director_name']])

print('\nInsight: Movies with High Votes but Lower Rating (popular but divisive):')
high_votes_lower_rating = movies_2023[
    (movies_2023['numVotes'] > movies_2023['numVotes'].quantile(0.75)) & 
    (movies_2023['averageRating'] < 6.5)
].nlargest(5, 'numVotes')
display(high_votes_lower_rating[['primaryTitle', 'averageRating', 'numVotes', 'director_name']])

## Q2: Most Popular Actor in 2023

**Approach**:
- Actor popularity = `SUM( (1/log(ordering+1)) × numVotes × averageRating )` (D008, D009)
- Billing weight gives more credit to lead actors
- numVotes = audience reach proxy
- averageRating = quality filter
- Filter principals to category in ['actor', 'actress']

In [ ]:
# Load principals data (needed for 2023 actor analysis)
print('Loading principals for 2023 actor analysis...')
import zipfile
SOURCE_DIR = os.path.join('..', 'source_data')

title_principals = pd.read_csv(
    os.path.join(SOURCE_DIR, 'title.principals.tsv.gz'),
    sep='\t', compression='gzip', na_values='\\N', low_memory=False
)
title_ratings = pd.read_csv(
    os.path.join(SOURCE_DIR, 'title.ratings.tsv.gz'),
    sep='\t', compression='gzip', na_values='\\N', low_memory=False
)
name_basics = pd.read_csv(
    os.path.join(SOURCE_DIR, 'name.basics.tsv.gz'),
    sep='\t', compression='gzip', na_values='\\N', low_memory=False
)

# Filter to actor/actress
title_principals = title_principals[
    title_principals['category'].isin(['actor', 'actress'])
].copy()
print(f'  Principals (actor/actress only): {len(title_principals):,} rows')

In [ ]:
# Get 2023 movie tconsts
tconsts_2023 = set(movies_2023['tconst'])

# Filter principals to 2023 movies
principals_2023 = title_principals[
    title_principals['tconst'].isin(tconsts_2023)
].merge(title_ratings, on='tconst', how='left')

print(f'Principals in 2023 movies: {len(principals_2023):,}')
print(f'Unique actors: {principals_2023["nconst"].nunique():,}')

# Compute billing weight (D009)
principals_2023['billing_weight'] = 1.0 / np.log(
    principals_2023['ordering'].astype(float) + 1
)

# Compute weighted score
principals_2023['weighted_score'] = (
    principals_2023['billing_weight'] * 
    principals_2023['numVotes'].fillna(0) * 
    principals_2023['averageRating'].fillna(0)
)

# Aggregate by actor
actor_2023 = principals_2023.groupby('nconst').agg(
    total_score=('weighted_score', 'sum'),
    num_movies=('tconst', 'nunique'),
    total_votes=('numVotes', 'sum'),
    avg_billing_order=('ordering', 'mean')
).reset_index()

# Add actor names
actor_names = name_basics[['nconst', 'primaryName']].drop_duplicates()
actor_2023 = actor_2023.merge(actor_names, on='nconst', how='left')
actor_2023 = actor_2023.sort_values('total_score', ascending=False)

print(f'\nTop 15 Most Popular Actors in 2023:')
display(actor_2023.head(15)[['primaryName', 'total_score', 'num_movies', 'total_votes', 'avg_billing_order']])

In [ ]:
# Bar chart: Top 15 actors
top15_actors = actor_2023.head(15)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(
    range(len(top15_actors)), 
    top15_actors['total_score'] / 1e6,  # Convert to millions
    color='steelblue', edgecolor='white'
)
ax.set_yticks(range(len(top15_actors)))
ax.set_yticklabels(top15_actors['primaryName'])
ax.invert_yaxis()
ax.set_xlabel('Weighted Popularity Score (millions)')
ax.set_title('Q2: Top 15 Most Popular Actors in 2023\n(by weighted billing × numVotes × rating)')

# Add value labels
for i, (idx, row) in enumerate(top15_actors.iterrows()):
    ax.text(row['total_score'] / 1e6 + 0.1, i, 
            f'{row["num_movies"]} films | {row["total_votes"]/1e6:.1f}M votes', 
            va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'q2_actor_popularity_2023.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q2_actor_popularity_2023.png')

In [ ]:
# Breakdown: Which films contributed to top actor's score?
print('\nBreakdown for Top 5 Actors — which films drove their popularity:')
print('='*80)

for nconst in top15_actors.head(5)['nconst']:
    actor_name = name_basics[name_basics['nconst'] == nconst]['primaryName'].values[0]
    actor_films = principals_2023[principals_2023['nconst'] == nconst].merge(
        movies_2023[['tconst', 'primaryTitle', 'averageRating', 'numVotes']], on='tconst'
    )
    print(f'\n{actor_name}:')
    display(actor_films[['primaryTitle', 'ordering', 'averageRating', 'numVotes', 'weighted_score']]
            .sort_values('weighted_score', ascending=False))

## Q3: User-Movie Preference Trends Over the Years

**Approach**:
- Genre frequency by decade (stacked area chart)
- Average runtime by decade (line chart)
- Vote count trend (audience engagement over time)
- Rating trend by decade

In [ ]:
# Create decade column
movies['decade'] = (movies['startYear'] // 10) * 10
movies['decade'] = movies['decade'].astype('Int64')

# Filter to reasonable decades (1920-2020)
movies_decades = movies[
    (movies['decade'] >= 1920) & (movies['decade'] <= 2020)
].copy()

print(f'Movies by decade (with >= {MIN_VOTE_THRESHOLD} votes):')
print(movies_decades.groupby('decade').size().to_string())

In [ ]:
# Genre analysis by decade
# Genres are comma-separated; explode them
movies_genres = movies_decades[['tconst', 'decade', 'genres']].dropna(subset=['genres']).copy()
movies_genres['genre_list'] = movies_genres['genres'].str.split(',')
movies_genres = movies_genres.explode('genre_list')
movies_genres['genre_list'] = movies_genres['genre_list'].str.strip()

# Count genres by decade
genre_counts = movies_genres.groupby(['decade', 'genre_list']).size().unstack(fill_value=0)

# Normalize to percentages
genre_pcts = genre_counts.div(genre_counts.sum(axis=1), axis=0) * 100

# Get top 8 genres overall
top_genres = genre_counts.sum().nlargest(8).index.tolist()
genre_pcts_top = genre_pcts[top_genres]

# Plot stacked area chart
fig, ax = plt.subplots(figsize=(14, 7))
genre_pcts_top.plot.area(ax=ax, alpha=0.8, stacked=True)
ax.set_xlabel('Decade')
ax.set_ylabel('Percentage of Movies')
ax.set_title('Q3: Genre Trends by Decade (Top 8 Genres)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xlim(1920, 2020)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'q3_genre_trends_decade.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q3_genre_trends_decade.png')

In [ ]:
# Runtime and rating trends by decade
decade_stats = movies_decades.groupby('decade').agg(
    avg_runtime=('runtimeMinutes', 'mean'),
    avg_rating=('averageRating', 'mean'),
    avg_votes=('numVotes', 'mean'),
    total_votes=('numVotes', 'sum'),
    num_movies=('tconst', 'count')
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Runtime trend
axes[0, 0].plot(decade_stats['decade'], decade_stats['avg_runtime'], 'o-', color='steelblue')
axes[0, 0].set_xlabel('Decade')
axes[0, 0].set_ylabel('Average Runtime (minutes)')
axes[0, 0].set_title('Average Movie Runtime by Decade')

# Rating trend
axes[0, 1].plot(decade_stats['decade'], decade_stats['avg_rating'], 'o-', color='coral')
axes[0, 1].set_xlabel('Decade')
axes[0, 1].set_ylabel('Average Rating')
axes[0, 1].set_title('Average Rating by Decade')

# Total votes trend (audience engagement)
axes[1, 0].bar(decade_stats['decade'], decade_stats['total_votes'] / 1e6, color='seagreen', width=8)
axes[1, 0].set_xlabel('Decade')
axes[1, 0].set_ylabel('Total Votes (millions)')
axes[1, 0].set_title('Audience Engagement by Decade (Total Votes)')

# Number of movies
axes[1, 1].bar(decade_stats['decade'], decade_stats['num_movies'], color='mediumpurple', width=8)
axes[1, 1].set_xlabel('Decade')
axes[1, 1].set_ylabel('Number of Movies')
axes[1, 1].set_title(f'Movies per Decade (with >= {MIN_VOTE_THRESHOLD} votes)')

plt.suptitle('Q3: Movie Preference Trends Over Decades', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'q3_trends_decades.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q3_trends_decades.png')

In [ ]:
# Heatmap: Genre popularity by decade (normalized within each decade)
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    genre_pcts_top.T, 
    annot=True, fmt='.0f', cmap='YlOrRd',
    ax=ax, linewidths=0.5
)
ax.set_xlabel('Decade')
ax.set_ylabel('Genre')
ax.set_title('Q3: Genre Popularity Heatmap (% of Movies per Decade)')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'q3_genre_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q3_genre_heatmap.png')

## Q4: Define a Hit Movie Metric

**Approach (D011)**: Two-mode hit metric

| Mode | Data Available | Formula | Coverage |
|------|---------------|---------|----------|
| **Full** | Budget + Revenue | `WR×0.4 + Reach×0.3 + ROI×0.3` | ~14,669 movies |
| **Partial** | Ratings + Votes only | `WR×0.6 + Reach×0.4` | ~481,442 movies |

**Components**:
- WR = IMDB Bayesian weighted rating (normalized 0-1)
- Reach = log(numVotes) normalized (0-1)
- ROI = (revenue - budget) / budget, capped and normalized (0-1)

**Validation**: Check that known hits rank in top decile.

In [ ]:
# Compute hit metric components

# 1. Weighted Rating (Bayesian)
C_all = movies['averageRating'].mean()
m_all = movies['numVotes'].quantile(0.25)
v_all = movies['numVotes']
R_all = movies['averageRating']
movies['weighted_rating'] = (v_all / (v_all + m_all)) * R_all + (m_all / (v_all + m_all)) * C_all

# Normalize to 0-1
wr_min, wr_max = movies['weighted_rating'].min(), movies['weighted_rating'].max()
movies['wr_normalized'] = (movies['weighted_rating'] - wr_min) / (wr_max - wr_min)

# 2. Audience Reach (log of numVotes, normalized)
movies['reach_log'] = np.log1p(movies['numVotes'])
reach_min, reach_max = movies['reach_log'].min(), movies['reach_log'].max()
movies['reach_normalized'] = (movies['reach_log'] - reach_min) / (reach_max - reach_min)

# 3. ROI (where available)
movies['roi'] = np.where(
    (movies['budget'] > 0) & (movies['revenue'] > 0),
    (movies['revenue'] - movies['budget']) / movies['budget'],
    np.nan
)
# Cap ROI at reasonable range (-100% to +1000%)
movies['roi_capped'] = movies['roi'].clip(-1, 10)
roi_min, roi_max = movies['roi_capped'].dropna().min(), movies['roi_capped'].dropna().max()
movies['roi_normalized'] = (movies['roi_capped'] - roi_min) / (roi_max - roi_min)

print('Hit Metric Components:')
print(f'  Weighted Rating (normalized): {movies["wr_normalized"].min():.3f} - {movies["wr_normalized"].max():.3f}')
print(f'  Audience Reach (normalized): {movies["reach_normalized"].min():.3f} - {movies["reach_normalized"].max():.3f}')
print(f'  ROI (normalized, {movies["roi_normalized"].notna().sum():,} movies): {movies["roi_normalized"].dropna().min():.3f} - {movies["roi_normalized"].dropna().max():.3f}')

In [ ]:
# Compute hit score (D011: two-mode)
# Full mode: WR×0.4 + Reach×0.3 + ROI×0.3
# Partial mode: WR×0.6 + Reach×0.4

# Full mode (where ROI available)
has_roi = movies['roi_normalized'].notna()
movies.loc[has_roi, 'hit_score'] = (
    0.4 * movies.loc[has_roi, 'wr_normalized'] + 
    0.3 * movies.loc[has_roi, 'reach_normalized'] + 
    0.3 * movies.loc[has_roi, 'roi_normalized']
)
movies.loc[has_roi, 'hit_mode'] = 'full'

# Partial mode (no ROI)
movies.loc[~has_roi, 'hit_score'] = (
    0.6 * movies.loc[~has_roi, 'wr_normalized'] + 
    0.4 * movies.loc[~has_roi, 'reach_normalized']
)
movies.loc[~has_roi, 'hit_mode'] = 'partial'

print(f'Hit Score computed for {len(movies):,} movies')
print(f'  Full mode (with ROI): {has_roi.sum():,} ({has_roi.mean()*100:.1f}%)')
print(f'  Partial mode (no ROI): {(~has_roi).sum():,} ({(~has_roi).mean()*100:.1f}%)')
print(f'  Hit score range: {movies["hit_score"].min():.3f} - {movies["hit_score"].max():.3f}')

In [ ]:
# Validation: Check known hits
print('Validation: Known Hits in Dataset')
print('='*80)

known_hits = [
    'The Dark Knight', 'Avengers: Endgame', 'Inception', 
    'The Shawshank Redemption', 'Pulp Fiction', 'Parasite',
    'Barbie', 'Oppenheimer', 'The Godfather', 'Titanic'
]

for title in known_hits:
    match = movies[movies['primaryTitle'].str.contains(title, case=False, na=False)]
    if len(match) > 0:
        row = match.iloc[0]
        print(f'{title}: hit_score={row["hit_score"]:.3f} ({row["hit_mode"]} mode)')
    else:
        print(f'{title}: NOT FOUND')

In [ ]:
# Distribution of hit scores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall distribution
axes[0].hist(movies['hit_score'], bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].axvline(movies['hit_score'].quantile(0.9), color='red', linestyle='--', 
                label=f'90th pctl: {movies["hit_score"].quantile(0.9):.3f}')
axes[0].set_xlabel('Hit Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Hit Scores')
axes[0].legend()

# By mode
for mode, color in [('full', 'coral'), ('partial', 'steelblue')]:
    subset = movies[movies['hit_mode'] == mode]
    axes[1].hist(subset['hit_score'], bins=30, alpha=0.6, color=color, 
                 label=f'{mode} mode (n={len(subset):,})')
axes[1].set_xlabel('Hit Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Hit Score by Mode (Full vs Partial)')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'q4_hit_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q4_hit_score_distribution.png')

In [ ]:
# Top 20 movies by hit score
print('\nTop 20 Movies by Hit Score (All Time):')
print('='*80)
top20_hits = movies.nlargest(20, 'hit_score')[
    ['primaryTitle', 'startYear', 'averageRating', 'numVotes', 'budget', 'revenue', 'hit_score', 'hit_mode']
]
display(top20_hits)

In [ ]:
# Save updated dataframe with hit scores
output_path = os.path.join(DATA_DIR, 'movies_with_hit_score.parquet')
movies.to_parquet(output_path, index=False)
print(f'Saved movies with hit scores to {os.path.abspath(output_path)}')

## Summary

**Q1**: Highest rated 2023 movie identified. Weighted rating (Bayesian) balances rating vs vote count. Scatter plot shows the rating-votes tradeoff.

**Q2**: Most popular actor in 2023 identified using weighted formula: `SUM((1/log(ordering+1)) × numVotes × averageRating)`. Billing weight ensures leads score higher than extras.

**Q3**: Genre trends show clear shifts over decades. Runtime, rating, and audience engagement trends visualized.

**Q4**: Hit metric defined with two-mode approach (D011). Full mode uses ROI where available; partial mode uses rating + reach. Validated against known hits.